In [7]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
import time
import os

# --- 1. Setup & Reliable Image Downloads ---
def download_image(url, filename):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'} # Pretend to be a browser
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print(f"Successfully downloaded {filename}")
        else:
            print(f"Failed to download {filename}. Status code: {response.status_code}")
    except Exception as e:
        print(f"Error downloading {filename}: {e}")

# Updated with very reliable Wikimedia direct links
image_urls = {
    "dog.jpg": "https://upload.wikimedia.org/wikipedia/commons/d/d9/Golden_Retriever_Hund_at_Work.jpg",
    "car.jpg": "https://upload.wikimedia.org/wikipedia/commons/a/af/2017_Audi_A4_Ultra_Front.jpg",
    "elephant.jpg": "https://upload.wikimedia.org/wikipedia/commons/3/37/African_Bush_Elephant.jpg",
    "pizza.jpg": "https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg",
    "airplane.jpg": "https://upload.wikimedia.org/wikipedia/commons/0/0b/Boeing_747-412_9V-SPQ_SIA_Heathrow.jpg",
    "ambiguous.jpg": "https://upload.wikimedia.org/wikipedia/commons/1/1f/Cloud_looks_like_dog.jpg",
    "blurry.jpg": "https://upload.wikimedia.org/wikipedia/commons/d/d3/Motion_blur_dog.jpg"
}

print("Downloading images...")
for name, url in image_urls.items():
    download_image(url, name)

Failed to download dog.jpg. Status code: 404
Failed to download car.jpg. Status code: 404
Failed to download elephant.jpg. Status code: 429
Successfully downloaded pizza.jpg
Failed to download airplane.jpg. Status code: 429
Failed to download ambiguous.jpg. Status code: 429
Failed to download blurry.jpg. Status code: 429


In [8]:
# --- 2. Define Model and Preprocessing ---
# Download ImageNet labels
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.split("\n")

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model50 = models.resnet50(weights='DEFAULT').eval()

In [10]:
# --- 3. Classification Function ---
def classify_image(image_path, model, top_k=5):
    try:
        img = Image.open(image_path).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0)
        
        with torch.no_grad():
            output = model(img_tensor)
            
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
        top_probs, top_idxs = torch.topk(probabilities, top_k)
        
        results = []
        for i in range(top_k):
            results.append({"class": labels[top_idxs[i].item()], "prob": top_probs[i].item()})
        return results
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None


In [11]:
# --- 4. Run Task 1 & 2 ---
test_images = ["dog.jpg", "car.jpg", "elephant.jpg", "pizza.jpg", "airplane.jpg"]

print("\n--- Classification Results ---")
for img_p in test_images:
    preds = classify_image(img_p, model50)
    if preds:
        print(f"\n{img_p}:")
        for i, p in enumerate(preds, 1):
            print(f"  {i}. {p['class']:25s} {p['prob']*100:.2f}%")


--- Classification Results ---
Error processing dog.jpg: cannot identify image file 'dog.jpg'
Error processing car.jpg: cannot identify image file 'car.jpg'
Error processing elephant.jpg: cannot identify image file 'elephant.jpg'

pizza.jpg:
  1. pizza                     63.84%
  2. broccoli                  0.31%
  3. soup bowl                 0.23%
  4. potpie                    0.22%
  5. bagel                     0.17%
Error processing airplane.jpg: cannot identify image file 'airplane.jpg'


In [13]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
import time
import os

# --- PRE-FIX: Ensure images are valid before starting Task 3 ---
def fix_image(filename, url):
    try:
        # We download even if it exists to overwrite the "broken" file causing your error
        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(url, headers=headers)
        with open(filename, 'wb') as f:
            f.write(r.content)
        # Verify it can be opened
        Image.open(filename).verify()
        print(f"Fixed/Verified: {filename}")
    except:
        print(f"Warning: Could not fix {filename}. You might need to delete it manually.")

# Using a very reliable direct link to fix the 'dog.jpg' error
fix_image("dog.jpg", "https://upload.wikimedia.org/wikipedia/commons/d/d9/Golden_Retriever_Hund_at_Work.jpg")

# --- Task 3: Compare Different ResNet Models ---
print("\n" + "="*65)
print(f"{'Model Name':<15} | {'Params (M)':<12} | {'Avg Inference (ms)':<18}")
print("-" * 65)

# 1. Initialize models
resnet_variants = {
    "ResNet-18": models.resnet18(weights='DEFAULT').eval(),
    "ResNet-50": models.resnet50(weights='DEFAULT').eval(),
    "ResNet-101": models.resnet101(weights='DEFAULT').eval()
}

# Use images you've already downloaded
test_images = ["dog.jpg", "car.jpg", "elephant.jpg", "pizza.jpg", "airplane.jpg"]

for name, model in resnet_variants.items():
    # a) Calculate parameters
    num_params = sum(p.numel() for p in model.parameters()) / 1e6
    
    total_time = 0
    count = 0
    
    for img_p in test_images:
        try:
            # b) Measure Inference time
            img = Image.open(img_p).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0)
            
            with torch.no_grad():
                start_time = time.perf_counter()
                _ = model(img_tensor)
                total_time += (time.perf_counter() - start_time)
                count += 1
        except Exception as e:
            # If an image is still broken, skip it instead of crashing
            continue
            
    if count > 0:
        avg_time_ms = (total_time / count) * 1000
        print(f"{name:<15} | {num_params:<12.2f} | {avg_time_ms:<18.2f}")
    else:
        print(f"{name:<15} | {num_params:<12.2f} | Error: No valid images found")

print("="*65)


Model Name      | Params (M)   | Avg Inference (ms)
-----------------------------------------------------------------
ResNet-18       | 11.69        | 139.64            
ResNet-50       | 25.56        | 135.86            
ResNet-101      | 44.55        | 241.09            


In [17]:
import os
import requests
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# --- FIX: Re-download corrupted Task 4 images ---
error_images_links = {
    "ambiguous.jpg": "https://upload.wikimedia.org/wikipedia/commons/1/1f/Cloud_looks_like_dog.jpg",
    "blurry.jpg": "https://upload.wikimedia.org/wikipedia/commons/d/d3/Motion_blur_dog.jpg"
}

headers = {'User-Agent': 'Mozilla/5.0'}

for filename, url in error_images_links.items():
    # Force delete existing broken file
    if os.path.exists(filename):
        os.remove(filename)
    
    # Download fresh copy
    try:
        r = requests.get(url, headers=headers)
        with open(filename, 'wb') as f:
            f.write(r.content)
        # Verify it can actually be opened
        with Image.open(filename) as img:
            img.verify()
        print(f" Successfully fixed {filename}")
    except Exception as e:
        print(f" Failed to fix {filename}: {e}")

# --- Task 4: Error Analysis ---
print("\n" + "="*50)
print("Task 4: Error Analysis Output")
print("="*50)

# Load model (using ResNet-50 as requested)
model50 = models.resnet50(weights='DEFAULT').eval()

# ImageNet labels (ensure this was loaded in previous cells)
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.split("\n")

for img_p in ["ambiguous.jpg", "blurry.jpg"]:
    try:
        img = Image.open(img_p).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0)
        
        with torch.no_grad():
            output = model50(img_tensor)
        
        probs = torch.nn.functional.softmax(output[0], dim=0)
        prob, idx = torch.max(probs, dim=0)
        
        print(f"Image: {img_p}")
        print(f"  -> Predicted Class: {labels[idx.item()]}")
        print(f"  -> Confidence:      {prob.item()*100:.2f}%")
        print("-" * 30)
    except Exception as e:
        print(f"Skipping {img_p} due to error: {e}")



 Failed to fix ambiguous.jpg: cannot identify image file 'ambiguous.jpg'
 Failed to fix blurry.jpg: cannot identify image file 'blurry.jpg'

Task 4: Error Analysis Output
Skipping ambiguous.jpg due to error: cannot identify image file 'ambiguous.jpg'
Skipping blurry.jpg due to error: cannot identify image file 'blurry.jpg'


In [19]:
import os
import requests
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time

# --- STEP 1: DEEP CLEAN & RELIABLE DOWNLOAD ---
image_urls = {
    "dog.jpg": "https://upload.wikimedia.org/wikipedia/commons/d/d9/Golden_Retriever_Hund_at_Work.jpg",
    "car.jpg": "https://upload.wikimedia.org/wikipedia/commons/a/af/2017_Audi_A4_Ultra_Front.jpg",
    "elephant.jpg": "https://upload.wikimedia.org/wikipedia/commons/3/37/African_Bush_Elephant.jpg",
    "pizza.jpg": "https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg",
    "airplane.jpg": "https://upload.wikimedia.org/wikipedia/commons/0/0b/Boeing_747-412_9V-SPQ_SIA_Heathrow.jpg"
}

print("Cleaning and downloading valid images...")
headers = {'User-Agent': 'Mozilla/5.0'}
valid_images = []

for filename, url in image_urls.items():
    if os.path.exists(filename): os.remove(filename) # Remove the corrupted file
    try:
        r = requests.get(url, headers=headers, timeout=10)
        with open(filename, 'wb') as f:
            f.write(r.content)
        with Image.open(filename) as img:
            img.verify() # Ensure it's a real image
        valid_images.append(filename)
        print(f"  [OK] {filename}")
    except:
        print(f"  [FAIL] Could not get {filename}")

# --- STEP 2: SETUP MODEL & TRANSFORM ---
model = models.resnet50(weights='DEFAULT').eval()
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- STEP 3: EXTRA TASK - BATCH PROCESSING ---
print("\n" + "="*50)
print("Extra Task: Batch Processing Performance")
print("="*50)

# Create a list of 10 images (duplicating our 5 valid ones)
batch_paths = valid_images * 2 

# A. Individual Processing (The slow way)
start_indiv = time.perf_counter()
for img_p in batch_paths:
    img = Image.open(img_p).convert('RGB')
    input_tensor = preprocess(img).unsqueeze(0)
    with torch.no_grad():
        _ = model(input_tensor)
time_indiv = time.perf_counter() - start_indiv

# B. Batch Processing (The fast way)
def run_batch_processing(image_paths, model):
    # Preprocess all images into a list
    tensors = [preprocess(Image.open(p).convert('RGB')) for p in image_paths]
    # Stack them into a single 4D tensor [BatchSize, Channels, H, W]
    batch_tensor = torch.stack(tensors)
    
    start_batch = time.perf_counter()
    with torch.no_grad():
        _ = model(batch_tensor)
    return time.perf_counter() - start_batch

time_batch = run_batch_processing(batch_paths, model)

# RESULTS
print(f"Number of images processed: {len(batch_paths)}")
print(f"Total time (Individual):    {time_indiv:.4f} seconds")
print(f"Total time (Batch):         {time_batch:.4f} seconds")
print(f"Speedup Factor:             {time_indiv / time_batch:.2f}x")


Cleaning and downloading valid images...
  [FAIL] Could not get dog.jpg
  [FAIL] Could not get car.jpg
  [OK] elephant.jpg
  [OK] pizza.jpg
  [FAIL] Could not get airplane.jpg

Extra Task: Batch Processing Performance
Number of images processed: 4
Total time (Individual):    0.6786 seconds
Total time (Batch):         0.3427 seconds
Speedup Factor:             1.98x
